# 🛒 Alura Agente — Mercado Central 24h

Agente de IA que responde preguntas en lenguaje natural sobre los documentos internos de **Mercado Central 24h** (supermercado 24/7): inventario, política de atención al cliente, FAQ, reglamento interno y manual de proveedores.

**Stack:** Python + LangChain + Gemini (Google) + FAISS.

**Cómo usar este notebook:**
1. Corré las celdas en orden, de arriba hacia abajo, una sola vez por sesión.
2. Cuando te pida la API Key de Gemini, generá una gratis en https://aistudio.google.com/apikey (no requiere tarjeta).
3. Al final, preguntale lo que quieras al agente.

> ⚠️ Importante: no reejecutes la celda 5 (creación del índice) varias veces seguidas "para probar" — cada corrida gasta cuota gratuita de embeddings (1000/día). Ejecutala una sola vez por sesión; de ahí en más, usá la función `preguntar(...)` todas las veces que quieras, sin costo adicional de cuota de embeddings.

## 1. Instalar dependencias

In [ ]:
!pip install -q -U langchain langchain-community langchain-google-genai langchain-huggingface langchain-text-splitters sentence-transformers pypdf pandas openpyxl faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.

## 2. Configurar la API Key de Gemini

Generá una gratis en https://aistudio.google.com/apikey y pegala cuando se te pida (no queda guardada en el notebook).

In [ ]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Pegá tu API Key de Gemini: ")

Pegá tu API Key de Gemini: ··········


## 3. Descargar los documentos de Mercado Central 24h

Se descargan automáticamente desde los links del curso: 4 PDFs + 1 Excel.

In [ ]:
import requests

archivos = {
    "faq.pdf": "https://cdn1.gnarususercontent.com.br/documents/6/internal/c0a37625-c8ee-44f2-885f-7b30480d3017.pdf",
    "politica_atencion.pdf": "https://cdn1.gnarususercontent.com.br/documents/6/internal/b9abdeaf-ffcb-46c4-8e1b-16935a594875.pdf",
    "reglamento_interno.pdf": "https://cdn1.gnarususercontent.com.br/documents/6/internal/ed99ddc6-bead-47fd-886b-c007c7e36885.pdf",
    "manual_proveedores.pdf": "https://cdn1.gnarususercontent.com.br/documents/6/internal/6ce88d4f-3c95-42c3-aef3-c156e859f000.pdf",
    "inventario.xlsx": "https://cdn1.gnarususercontent.com.br/documents/6/internal/e4dc3cc4-55d5-453b-bc68-3f5ef090fc7f.xlsx",
}

for nombre, url in archivos.items():
    r = requests.get(url)
    r.raise_for_status()
    with open(nombre, "wb") as f:
        f.write(r.content)
    print(f"✅ Descargado: {nombre} ({len(r.content) / 1024:.1f} KB)")

✅ Descargado: faq.pdf (919.6 KB)
✅ Descargado: politica_atencion.pdf (731.8 KB)
✅ Descargado: reglamento_interno.pdf (718.6 KB)
✅ Descargado: manual_proveedores.pdf (801.6 KB)
✅ Descargado: inventario.xlsx (99.8 KB)


## 4. Cargar y trocear los documentos

Los 4 PDFs se leen como texto. El Excel de inventario se agrupa en bloques de filas (no fila por fila) para no gastar cuota de más en el paso de embeddings.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import pandas as pd

todos_documentos = []

# --- PDFs ---
pdfs = ["faq.pdf", "politica_atencion.pdf", "reglamento_interno.pdf", "manual_proveedores.pdf"]
for pdf in pdfs:
    loader = PyPDFLoader(pdf)
    paginas = loader.load()
    todos_documentos.extend(paginas)
    print(f"📄 {pdf}: {len(paginas)} página(s) cargada(s)")

# --- Excel de inventario (agrupado en bloques de 25 filas) ---
GRUPO_FILAS = 25
df = pd.read_excel("inventario.xlsx")
print(f"\n📊 inventario.xlsx: {len(df)} filas, columnas: {list(df.columns)}")

for i in range(0, len(df), GRUPO_FILAS):
    bloque = df.iloc[i:i + GRUPO_FILAS]
    texto_bloque = bloque.to_string(index=False)
    todos_documentos.append(Document(page_content=texto_bloque, metadata={"source": "inventario.xlsx"}))

# --- Trocear todo en fragmentos manejables ---
splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
chunks = splitter.split_documents(todos_documentos)

print(f"\n✅ Total de fragmentos generados: {len(chunks)}")

/tmp/ipykernel_1501/2813475082.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


📄 faq.pdf: 37 página(s) cargada(s)
📄 politica_atencion.pdf: 28 página(s) cargada(s)
📄 reglamento_interno.pdf: 28 página(s) cargada(s)
📄 manual_proveedores.pdf: 31 página(s) cargada(s)

📊 inventario.xlsx: 200 filas, columnas: ['SKU', 'Cód. de Barras (EAN)', 'Descripción', 'Marca', 'Categoría', 'Subcategoría', 'UN', 'Ubicación', 'Stock Actual', 'Stock Mínimo', 'Stock Máximo', 'Lote', 'Fecha de Fabricación', 'Fecha de Vencimiento', 'Costo Unitario', 'Precio de Venta Unitario', 'Proveedor Principal', 'Tiempo de Reposición']

✅ Total de fragmentos generados: 243


## 5. Crear embeddings y la base vectorial (FAISS)

Convierte cada fragmento de texto en un vector numérico, para poder buscar por significado (no solo por palabras exactas).

⚠️ Ejecutá esta celda **una sola vez por sesión**. Si la reejecutás varias veces, va a volver a gastar cuota gratuita de embeddings.

In [ ]:
!pip install -q -U sentence-transformers langchain-huggingface

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("✅ Base vectorial creada correctamente (sin cuota, sin límites)")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Base vectorial creada correctamente (sin cuota, sin límites)


In [ ]:
!pip show langchain
!pip list | grep -i langchain

Name: langchain
Version: 1.3.13
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
langchain                             1.3.13
langchain-classic                     1.0.8
langchain-community                   0.4.2
langchain-core                        1.4.9
langchain-google-genai                4.2.7
langchain-huggingface                 1.2.2
langchain-protocol                    0.0.18
langchain-text-splitters              1.1.2


## 6. Crear el agente (LLM + cadena de preguntas y respuestas)

In [ ]:
!pip install -q -U langchain

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0)

system_prompt = """Sos el asistente virtual de Mercado Central 24h. Respondé preguntas de colaboradores o clientes
basándote únicamente en el contexto de los documentos internos que se te provee (inventario, política de atención,
FAQ, reglamento interno, manual de proveedores).
Si la respuesta no está en el contexto, decí claramente que no encontraste esa información en los documentos, no inventes datos.
Respondé siempre en español, de forma clara y directa, citando de qué documento sale la información si es relevante.

Contexto:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
qa_chain = create_retrieval_chain(retriever, question_answer_chain)

print("✅ Agente listo")

✅ Agente listo


## 7. Función para preguntar

In [ ]:
def preguntar(pregunta: str):
    resultado = qa_chain.invoke({"input": pregunta})
    print("PREGUNTA:", pregunta)
    print("\nRESPUESTA:", resultado["answer"])
    print("\n--- Fuentes usadas ---")
    for doc in resultado["context"]:
        print("-", doc.metadata.get("source", "desconocida"))
    return resultado

## 8. Probar el agente

Algunos ejemplos de preguntas sobre los documentos de Mercado Central 24h. Probá también con las tuyas.

In [ ]:
_ = preguntar("¿Cuál es el plazo para devolver carnes y pescados frescos?")

PREGUNTA: ¿Cuál es el plazo para devolver carnes y pescados frescos?

RESPUESTA: De acuerdo con la política de devoluciones de **Mercado Central 24h**, el plazo para realizar el cambio de **carnes y pescados frescos** es el **mismo día de la compra, antes de las 22:00 horas**.

Esta información se encuentra detallada en la sección de políticas de productos perecederos del manual de atención al cliente.

--- Fuentes usadas ---
- faq.pdf
- reglamento_interno.pdf
- politica_atencion.pdf
- politica_atencion.pdf


In [ ]:
_ = preguntar("¿Cuántos días de vacaciones tengo con 5 años de antigüedad?")

PREGUNTA: ¿Cuántos días de vacaciones tengo con 5 años de antigüedad?

RESPUESTA: No encontré información específica sobre la cantidad de días de vacaciones que corresponden según los años de antigüedad en los documentos proporcionados.

Los documentos del **Reglamento Interno (Sección B4. Nómina e Incidencias)** solo detallan el procedimiento para solicitarlas, la política de acumulación y que la antigüedad otorga prelación (prioridad) al elegir las fechas, pero no incluyen la tabla de días por año trabajado. Te sugiero consultar directamente con el área de Capital Humano para obtener el dato exacto según tu antigüedad.

--- Fuentes usadas ---
- faq.pdf
- faq.pdf
- politica_atencion.pdf
- faq.pdf


In [ ]:
_ = preguntar("¿Cómo funciona el programa Cliente VIP Central y cuáles son sus niveles?")

PREGUNTA: ¿Cómo funciona el programa Cliente VIP Central y cuáles son sus niveles?

RESPUESTA: El programa **"Cliente VIP Central"** es un sistema de fidelidad gratuito diseñado para que los clientes acumulen "Pesos Central" por sus compras y accedan a beneficios exclusivos. Según el documento **A3. PROGRAMA DE FIDELIDAD "CLIENTE VIP CENTRAL"**, el programa funciona de la siguiente manera:

### **Niveles de Membresía**
El programa cuenta con tres niveles que se alcanzan según el monto de compra acumulado:

1.  **Nivel Plata (Entrada):**
    *   **Requisito:** Se obtiene automáticamente al registrarse, sin compra mínima.
    *   **Beneficios:** Acumulación del **2% en "Pesos Central"**, precios exclusivos (etiqueta plateada), notificaciones anticipadas de ofertas y acceso a contenido exclusivo en la App.

2.  **Nivel Oro:**
    *   **Requisito:** Acumular compras por **$5,000 MXN o más** en un periodo de 3 meses consecutivos.
    *   **Mantenimiento:** Se mantiene por 6 meses realizando

## 9. Modo interactivo (opcional)

Corré esta celda para hacer preguntas libremente. Escribí `salir` para terminar.

In [ ]:
while True:
    pregunta = input("\nHacé una pregunta ('salir' para terminar): ")
    if pregunta.strip().lower() == "salir":
        break
    preguntar(pregunta)


Hacé una pregunta ('salir' para terminar): hola
PREGUNTA: hola

RESPUESTA: ¡Hola! Soy el asistente virtual de Mercado Central 24h. ¿En qué puedo ayudarte hoy?

Puedo brindarte información basada en nuestros documentos oficiales sobre:
*   **Programa VIP:** Acumulación y canje de puntos, beneficios y vigencia.
*   **Atención a clientes con necesidades especiales:** Accesibilidad física y atención prioritaria.
*   **Información corporativa y de contacto:** Ubicación, teléfonos y correos electrónicos.
*   **Procedimientos internos:** Como la entrega de turno y reportes de incidencias.

¿Tienes alguna duda específica sobre estos temas?

--- Fuentes usadas ---
- politica_atencion.pdf
- politica_atencion.pdf
- reglamento_interno.pdf
- politica_atencion.pdf

Hacé una pregunta ('salir' para terminar): salir


---
## Próximo paso: deploy en OCI

Una vez que este agente funcione bien acá, el siguiente paso es envolver esta lógica en un script/API (por ejemplo con Flask o FastAPI) y desplegarla en una instancia de OCI Compute. Ver el `README.md` del repositorio para la guía paso a paso.